# K-Nearest Neighbors (KNN) from Scratch — A Hands-On Tutorial

*Understand, build, and evaluate one of the simplest yet most intuitive machine-learning algorithms — no scikit-learn model required.*

---

## What this notebook teaches

This tutorial takes the compact `knn.py` script (a pure-NumPy implementation of the **K-Nearest Neighbors** classifier) and turns it into a complete, beginner-friendly learning experience. You will:

- Understand the **intuition** behind KNN and when to use it.
- Read and understand a **from-scratch implementation** (no ML library does the learning for us).
- Walk through the **full machine-learning workflow**: load → explore → preprocess → split → train → predict → evaluate → interpret.
- Produce useful **visualizations**: feature distributions, a correlation heatmap, a decision boundary, and a confusion matrix.
- Learn how to **choose `k`** and how to make single and batch predictions.

## The machine-learning task

KNN here solves a **multi-class classification** problem: given the measurements of an iris flower, predict which of **three species** it belongs to (*setosa*, *versicolor*, *virginica*). Classification means the output is a **category (label)**, not a number.

## Learning outcomes

By the end you will be able to:

1. Explain how KNN classifies a new point using its nearest neighbors.
2. Describe why feature **scaling** matters for distance-based models.
3. Evaluate a classifier with **accuracy, precision, recall, F1-score,** and a **confusion matrix**.
4. Visualize a **decision boundary** and reason about **overfitting vs. underfitting** through the choice of `k`.

## 1. The concept, in plain language

**K-Nearest Neighbors** is about the simplest idea in machine learning:

> *"To classify something new, look at the examples most similar to it and let them vote."*

There is no training in the usual sense — KNN simply **memorizes** the training data. When a new example arrives, it:

1. Measures the **distance** from the new point to every stored training point.
2. Picks the **`k` closest** ones (the *nearest neighbors*).
3. Takes a **majority vote** of their labels and assigns the winning label.

Because it does the work only when a prediction is requested, KNN is called a **lazy learner** or **instance-based** method.

### A real-world analogy

Imagine you move to a new neighborhood and want to guess whether a house is *expensive* or *affordable*. A natural approach: look at the **5 nearest houses** and see how most of them are priced. If 4 of the 5 closest houses are expensive, you'd guess this one is expensive too. That is exactly KNN with `k = 5`.

### Key ideas to keep in mind

- **`k` is a knob you choose.** Small `k` (e.g. 1) follows the data very closely and can be noisy (**overfitting**). Large `k` smooths decisions but can blur real boundaries (**underfitting**).
- **Distance depends on scale.** A feature measured in the thousands will dominate one measured in fractions unless we **standardize** the features first.
- **No explicit model / no parameters are learned** — the training set *is* the model.

## 2. The machine-learning workflow we'll follow

Every supervised-learning project follows roughly the same steps. We'll map each one to a section below:

| Step | What happens | Where in this notebook |
|------|--------------|------------------------|
| Data loading | Read the dataset into memory | Section 5 |
| Data exploration | Look at shapes, stats, class balance | Section 6 |
| Preprocessing | Scale features so distances are fair | Section 7 |
| Train/test split | Hold out data to test honestly | Section 7 |
| Model training | `fit()` — here, just memorize | Section 8 |
| Prediction | `predict()` — vote among neighbors | Section 8 |
| Evaluation | Accuracy, precision/recall, confusion matrix | Section 9 |
| Interpretation | Decision boundary, choosing `k` | Sections 10–11 |

```mermaid
flowchart LR
    A[Load data] --> B[Explore]
    B --> C[Preprocess / Scale]
    C --> D[Train / Test split]
    D --> E[Fit KNN - memorize]
    E --> F[Predict - neighbor vote]
    F --> G[Evaluate]
    G --> H[Interpret & tune k]
```

## 3. Setup — import the libraries we need

We rely on a small, standard stack:

- **NumPy** — fast array math (distances, sorting).
- **pandas** — tabular exploration of the dataset.
- **matplotlib / seaborn** — charts (distributions, heatmaps, decision boundary).
- **scikit-learn** — *only* for the dataset, the train/test split, and evaluation metrics. **The KNN model itself is written by hand.**

> If a package is missing, install it with `pip install numpy pandas matplotlib seaborn scikit-learn`.

In [1]:
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
)

# Make plots look clean and results reproducible
sns.set_theme(style="whitegrid")
RANDOM_STATE = 1234
np.random.seed(RANDOM_STATE)
%matplotlib inline

## 4. The KNN algorithm, from scratch

Below is the heart of the tutorial — the exact algorithm from `knn.py`, kept faithful to the original with a few explanatory comments added. Read it as three small pieces:

1. **`euclidean_distance`** — the straight-line distance between two points in feature space:

$$d(\mathbf{x}_1, \mathbf{x}_2) = \sqrt{\sum_i (x_{1,i} - x_{2,i})^2}$$

2. **`fit`** — for KNN this is trivial: just **store** the training data. (This is why KNN is a *lazy* learner.)

3. **`predict` / `_predict`** — for each new point: compute all distances, take the `k` smallest, and return the **most common label** among those neighbors.

In [2]:
def euclidean_distance(x1, x2):
    """Straight-line (L2) distance between two feature vectors."""
    return np.sqrt(np.sum((x1 - x2) ** 2))


class KNN:
    """K-Nearest Neighbors classifier implemented from scratch.

    Parameters
    ----------
    k : int
        How many nearest neighbors vote on each prediction.
    """

    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        # "Training" is just memorizing the data (lazy learning).
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        # Predict a label for every row in X.
        y_pred = [self._predict(x) for x in X]
        return np.array(y_pred)

    def _predict(self, x):
        # 1) distance from x to every training example
        distances = [euclidean_distance(x, x_train) for x_train in self.X_train]
        # 2) indices of the k closest training points
        k_idx = np.argsort(distances)[: self.k]
        # 3) labels of those k neighbors
        k_neighbor_labels = [self.y_train[i] for i in k_idx]
        # 4) majority vote -> most common label wins
        most_common = Counter(k_neighbor_labels).most_common(1)
        return most_common[0][0]

### Step-by-step: what just happened?

- **`euclidean_distance`** uses NumPy to compute $\sqrt{\sum (x_1 - x_2)^2}$ in one vectorized line. This is the notion of "similarity" KNN relies on — smaller distance means more similar.
- **`fit`** stores `X` and `y`. Notice there is **no math and no loop** — KNN postpones all work until prediction time.
- **`_predict`** is the core:
  - `np.argsort(distances)` returns the indices that would sort the distances from smallest to largest; `[: self.k]` keeps the `k` nearest.
  - `Counter(...).most_common(1)` finds the label that appears most often among those neighbors — the **majority vote**.
- **`predict`** simply applies `_predict` to each incoming row.

> **Design note:** this implementation is intentionally simple and readable ($O(n)$ distance computations per query). For large datasets you'd use spatial structures (KD-trees / Ball-trees) as scikit-learn does — but the *logic* is identical.

### Picture the distance — Euclidean distance is just Pythagoras

The `euclidean_distance` function is the heart of KNN, so it helps to *see* what it measures. For two points in 2D, the straight-line distance between them is the **hypotenuse** of a right triangle whose legs are the horizontal gap (`dx`) and the vertical gap (`dy`):

$$d = \sqrt{dx^2 + dy^2}$$

The chart below plots two points, draws the triangle, and confirms the famous **3-4-5** case: a horizontal gap of 4 and a vertical gap of 3 give a distance of exactly 5. In higher dimensions the same idea holds — we just sum the squared gaps across *all* features before taking the square root.

In [ ]:
# Two example points in 2D feature space
p1 = np.array([1.0, 1.0])
p2 = np.array([5.0, 4.0])

dx, dy = p2[0] - p1[0], p2[1] - p1[1]      # horizontal and vertical gaps
dist = np.sqrt(dx**2 + dy**2)              # Euclidean distance (hypotenuse)

fig, ax = plt.subplots(figsize=(6.5, 6))

# the two points
ax.scatter(*p1, s=120, color="#1f77b4", zorder=5)
ax.scatter(*p2, s=120, color="#d62728", zorder=5)
ax.annotate("P1 (1, 1)", p1, textcoords="offset points", xytext=(-45, -5), fontsize=11)
ax.annotate("P2 (5, 4)", p2, textcoords="offset points", xytext=(10, 0), fontsize=11)

# the right triangle: horizontal leg (dx), vertical leg (dy), hypotenuse (distance)
ax.plot([p1[0], p2[0]], [p1[1], p1[1]], "k--", lw=1.5)   # bottom leg  (dx)
ax.plot([p2[0], p2[0]], [p1[1], p2[1]], "k--", lw=1.5)   # right leg   (dy)
ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="green", lw=2.5)  # hypotenuse

# a small square to mark the 90-degree corner
ax.plot([p2[0]-0.3, p2[0]-0.3, p2[0]], [p1[1], p1[1]+0.3, p1[1]+0.3], "k-", lw=1)

# labels for each side
ax.text((p1[0]+p2[0])/2, p1[1]-0.35, f"dx = {dx:.0f}", ha="center", color="black", fontsize=11)
ax.text(p2[0]+0.15, (p1[1]+p2[1])/2, f"dy = {dy:.0f}", va="center", color="black", fontsize=11)
ax.text((p1[0]+p2[0])/2 - 0.5, (p1[1]+p2[1])/2 + 0.35,
        f"distance = {dist:.2f}", color="green", fontsize=12, rotation=37, fontweight="bold")

ax.set_title("Euclidean distance = the hypotenuse (Pythagoras)\n"
             r"$d=\sqrt{dx^2 + dy^2}=\sqrt{4^2+3^2}=5$")
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
ax.set_xlim(0, 6.5)
ax.set_ylim(0, 5.5)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"dx = {dx:.0f},  dy = {dy:.0f}")
print(f"Euclidean distance between P1 and P2 = {dist:.2f}")
print("Matches our euclidean_distance():", np.isclose(dist, euclidean_distance(p1, p2)))

## 5. Data loading

The original script trains on the classic **Iris** dataset, which ships with scikit-learn — so there is **no file to download**. It contains **150 flowers**, 50 from each of three species, described by **4 numeric features** (all in centimeters):

- sepal length, sepal width, petal length, petal width

The **target** `y` is the species, encoded as `0`, `1`, `2`.

In [ ]:
iris = datasets.load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print("Feature matrix X:", X.shape)   # (150 samples, 4 features)
print("Target vector y:", y.shape)    # (150 labels,)
print("Features:", feature_names)
print("Classes: ", list(target_names))

## 6. Data exploration

Before modeling, always *look* at the data. We convert it to a pandas `DataFrame` so we can inspect it as a table, check summary statistics, and confirm the classes are **balanced** (equally represented).

In [ ]:
df = pd.DataFrame(X, columns=feature_names)
df["species"] = pd.Categorical.from_codes(y, target_names)

print("First 5 rows:")
display(df.head())

print("\nSummary statistics:")
display(df.describe())

print("\nSamples per class (balanced?):")
print(df["species"].value_counts())

### Visual 1 — Feature distributions per species

Histograms show how each feature is distributed for each species. Notice that **petal length** and **petal width** separate the species almost perfectly, while the sepal features overlap more. Good separation = easier classification.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, feature in zip(axes.ravel(), feature_names):
    for species in target_names:
        subset = df[df["species"] == species]
        ax.hist(subset[feature], bins=12, alpha=0.6, label=species)
    ax.set_title(feature)
    ax.set_xlabel("cm")
    ax.set_ylabel("count")
    ax.legend()
fig.suptitle("Feature distributions by species", fontsize=14)
fig.tight_layout()
plt.show()

### Visual 2 — Correlation heatmap

The heatmap shows how strongly the four features move together. **Petal length** and **petal width** are highly correlated (~0.96): flowers with long petals tend to have wide petals too. Highly correlated features carry overlapping information.

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(
    df[feature_names].corr(),
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    fmt=".2f",
)
plt.title("Feature correlation heatmap")
plt.tight_layout()
plt.show()

### Visual 3 — Pairwise scatter (how separable are the classes?)

A scatter of **petal length vs. petal width**, colored by species, makes it obvious why KNN works so well here: the three species form fairly distinct clusters, so "nearest neighbors" usually share the same label.

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=df,
    x="petal length (cm)",
    y="petal width (cm)",
    hue="species",
    s=60,
)
plt.title("Petal length vs. petal width")
plt.tight_layout()
plt.show()

## 7. Preprocessing + train/test split

Two important steps before training:

**a) Train/test split.** We hold out 20% of the data as a **test set**. The model never sees it during training, so measuring accuracy on it gives an *honest* estimate of real-world performance.

**b) Feature scaling (an improvement over the original script).** KNN is **distance-based**, so features on larger numeric ranges would unfairly dominate the distance. We **standardize** each feature to mean 0 and standard deviation 1 with `StandardScaler`.

> **Why fit the scaler on training data only?** To avoid *data leakage*. The scaler learns the mean/std from the **training set**, then applies the same transform to the test set — mimicking how we'd treat genuinely unseen data.

*The original `knn.py` skipped scaling. Iris features are already on similar scales, so results barely change — but scaling is the correct habit for distance-based models and we adopt it here.*

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # learn mean/std on TRAIN
X_test_scaled = scaler.transform(X_test)        # apply same transform to TEST

print("Train samples:", X_train_scaled.shape[0])
print("Test samples: ", X_test_scaled.shape[0])

## 8. Train the model and make predictions

Now we use our hand-written `KNN`. "Training" just stores the data; the real computation happens in `predict`, where each test point votes among its `k = 3` nearest neighbors.

In [ ]:
clf = KNN(k=3)
clf.fit(X_train_scaled, y_train)

predictions = clf.predict(X_test_scaled)

acc = accuracy_score(y_test, predictions)
print(f"Test accuracy: {acc:.3f}  ({acc * 100:.1f}% correct)")

## 9. Model evaluation and interpretation

Accuracy alone can hide problems (e.g., a model that ignores a rare class). For classification we also look at **precision, recall, F1-score**, and the **confusion matrix**.

**Beginner-friendly definitions** (per class):

- **Accuracy** — overall fraction of correct predictions.
- **Precision** — of everything predicted as class *C*, how much really was *C*? (*Avoids false alarms.*)
- **Recall** — of all the true *C* examples, how many did we catch? (*Avoids misses.*)
- **F1-score** — the harmonic mean of precision and recall; a single balanced number.
- **Confusion matrix** — a table of *actual vs. predicted*; the diagonal are correct predictions, off-diagonal are mistakes.

In [ ]:
print(classification_report(y_test, predictions, target_names=target_names))

### Visual 4 — Confusion matrix

Each row is the **true** species; each column is the **predicted** species. A perfect classifier has all counts on the diagonal. Any off-diagonal cell tells you exactly which species got confused for which.

In [ ]:
cm = confusion_matrix(y_test, predictions)

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()

## 10. Visualizing the decision boundary

A **decision boundary** shows how the model splits the feature space into regions — every point in a colored region would be classified as that region's species. Because we can only draw in 2D, we train a second KNN using just the **two most informative features** (petal length & width).

Watch how the colored regions wrap around the natural clusters. That flexible, data-shaped boundary is the signature of KNN.

In [ ]:
from matplotlib.colors import ListedColormap

# Use only petal length (col 2) and petal width (col 3) so we can plot in 2D
X2 = X[:, 2:4]
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.2, random_state=RANDOM_STATE
)

knn2 = KNN(k=5)
knn2.fit(X2_train, y2_train)

# Build a grid of points covering the feature space
h = 0.05
x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid = np.c_[xx.ravel(), yy.ravel()]

# Classify every grid point, then reshape to color the background
Z = knn2.predict(grid).reshape(xx.shape)

cmap_bg = ListedColormap(["#FFDDDD", "#DDFFDD", "#DDDDFF"])
cmap_pts = ListedColormap(["#FF0000", "#00AA00", "#0000FF"])

plt.figure(figsize=(7, 5.5))
plt.contourf(xx, yy, Z, cmap=cmap_bg, alpha=0.6)
plt.scatter(X2[:, 0], X2[:, 1], c=y, cmap=cmap_pts, edgecolor="k", s=45)
plt.xlabel("petal length (cm)")
plt.ylabel("petal width (cm)")
plt.title("KNN decision boundary (k = 5)")
plt.tight_layout()
plt.show()

## 11. Choosing `k` — the key hyperparameter

`k` controls the **bias–variance trade-off**:

- **Small `k` (e.g. 1)** — boundary hugs every point → sensitive to noise (**high variance / overfitting**).
- **Large `k`** — boundary is very smooth → may ignore real structure (**high bias / underfitting**).

Let's evaluate accuracy across a range of `k` values and pick a good one.

In [ ]:
k_values = range(1, 21)
accuracies = []
for k in k_values:
    model = KNN(k=k)
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    accuracies.append(accuracy_score(y_test, preds))

best_k = k_values[int(np.argmax(accuracies))]

plt.figure(figsize=(8, 4.5))
plt.plot(list(k_values), accuracies, marker="o")
plt.axvline(best_k, color="red", linestyle="--", label=f"best k = {best_k}")
plt.xlabel("k (number of neighbors)")
plt.ylabel("test accuracy")
plt.title("How accuracy changes with k")
plt.xticks(list(k_values))
plt.legend()
plt.tight_layout()
plt.show()

print(f"Best k on this split: {best_k} (accuracy = {max(accuracies):.3f})")

## 12. Animations to build intuition

Static plots are great, but *watching* the model change makes the ideas click. Below are two short animations built with **`matplotlib.animation`** and shown inline as an interactive player (via `to_jshtml()`), so they work anywhere — no extra video tools (like `ffmpeg`) required.

> These cells recompute the model several times, so they may take a few seconds to render. Press the ▶ play button under each animation. If you prefer a saved GIF, see the note in Section 17.

### 12.1 The decision boundary as `k` grows

This animation sweeps `k` from very small to large and redraws the decision regions each time. Watch the boundary go from **jagged and overfit** (small `k`, reacting to every single point) to **smooth and general** (large `k`).

In [ ]:
from matplotlib import animation
from IPython.display import HTML

# Sweep k from small (overfit) to large (smooth) on the 2D petal features
k_frames = [1, 3, 5, 7, 10, 15, 20, 30]

# A coarse grid keeps the animation quick to build
step = 0.2
gx_min, gx_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
gy_min, gy_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
gx, gy = np.meshgrid(np.arange(gx_min, gx_max, step),
                     np.arange(gy_min, gy_max, step))
grid_pts = np.c_[gx.ravel(), gy.ravel()]

# Pre-compute the decision regions for every k (this is the slow part)
region_frames = []
for k in k_frames:
    m = KNN(k=k)
    m.fit(X2_train, y2_train)
    region_frames.append(m.predict(grid_pts).reshape(gx.shape))

fig, ax = plt.subplots(figsize=(7, 5.5))

def draw_boundary(i):
    ax.clear()
    ax.contourf(gx, gy, region_frames[i], cmap=cmap_bg, alpha=0.6)
    ax.scatter(X2[:, 0], X2[:, 1], c=y, cmap=cmap_pts, edgecolor="k", s=40)
    ax.set_title(f"KNN decision boundary — k = {k_frames[i]}")
    ax.set_xlabel("petal length (cm)")
    ax.set_ylabel("petal width (cm)")

anim = animation.FuncAnimation(fig, draw_boundary, frames=len(k_frames), interval=1200)
plt.close(fig)  # prevent a duplicate static figure
HTML(anim.to_jshtml())

**What to notice:** at `k = 1` the colored regions form little islands around individual points — the model reacts to noise (**overfitting**). As `k` increases the boundary straightens out and becomes more stable. Push `k` too high and it would start to ignore the real gap between species (**underfitting**). This is the **bias–variance trade-off** in motion.

### 12.2 Classifying a single point, step by step

Here we drop a single **query point** (the yellow star) into the feature space and reveal its neighbors one at a time. The dashed circle grows to include the `k` nearest training points, and the title tallies their votes and shows the current winning class.

In [ ]:
# Pick a query point near the tricky versicolor/virginica border
query = np.array([4.8, 1.5])
q_dists = np.array([euclidean_distance(query, p) for p in X2_train])
order = np.argsort(q_dists)
K_MAX = 15

fig, ax = plt.subplots(figsize=(7, 5.5))

def reveal_neighbors(frame):
    k = frame + 1
    ax.clear()
    # all training points, faint
    ax.scatter(X2_train[:, 0], X2_train[:, 1], c=y2_train, cmap=cmap_pts,
               s=35, alpha=0.4, edgecolor="none")
    # the k nearest, ringed in black
    nn = order[:k]
    ax.scatter(X2_train[nn, 0], X2_train[nn, 1], facecolors="none",
               edgecolors="black", s=150, linewidths=1.6)
    # dashed circle out to the k-th neighbor
    radius = q_dists[order[k - 1]]
    ax.add_patch(plt.Circle(query, radius, color="gray", fill=False, ls="--"))
    # the query point (yellow star)
    ax.scatter(*query, marker="*", s=320, c="yellow", edgecolor="k", zorder=5)
    # tally the votes among the k neighbors
    votes = Counter(y2_train[nn])
    winner = votes.most_common(1)[0][0]
    tally = "  ".join(f"{target_names[c]}={n}" for c, n in sorted(votes.items()))
    ax.set_title(f"k = {k}    votes: {tally}    ->  {target_names[winner]}")
    ax.set_xlabel("petal length (cm)")
    ax.set_ylabel("petal width (cm)")
    ax.set_xlim(gx_min, gx_max)
    ax.set_ylim(gy_min, gy_max)

anim2 = animation.FuncAnimation(fig, reveal_neighbors, frames=K_MAX, interval=900)
plt.close(fig)  # avoid a duplicate static figure
HTML(anim2.to_jshtml())

**What to notice:** with very small `k` the prediction can flip depending on one or two nearby points. As the circle grows to include more neighbors, the vote stabilizes. This is exactly what `KNN._predict` does internally — the animation just makes the "find the neighbors, then vote" process visible.

### 12.3 Worked example — inspect the neighbors by hand

Animations show the idea; this example shows the **actual numbers**. We take one flower from the test set, list its `k` nearest training neighbors with their distances and species, then tally the vote — the same computation the model performs internally.

In [ ]:
# Take one test flower and inspect the vote by hand
i = 0
x_query = X_test_scaled[i]
dists = np.array([euclidean_distance(x_query, xt) for xt in X_train_scaled])
k = 5
nn_idx = np.argsort(dists)[:k]

print(f"Test flower #{i}  (true species: {target_names[y_test[i]]})\n")
print(f"{k} nearest training neighbors:")
for rank, j in enumerate(nn_idx, start=1):
    print(f"  {rank}. distance = {dists[j]:.3f}  ->  {target_names[y_train[j]]}")

votes = Counter(y_train[nn_idx])
print("\nVote tally:", {target_names[c]: n for c, n in votes.items()})
print("Model prediction:", target_names[votes.most_common(1)[0][0]])

### Interpreting this example

- The neighbors are sorted by distance, so neighbor #1 is the most similar training flower.
- The **majority species** among the 5 neighbors becomes the prediction — here it should match the true species because the classes are well separated.
- If the neighbors were a mix of species (which happens near the versicolor/virginica border), the prediction would be less certain — a good place to report a confidence score (see Exercise 1).

## 13. Practical examples — using the trained model

Once trained, the model is easy to use. Remember: any **new input must be scaled with the same `scaler`** we fit on the training data.

### Example A — a single prediction

We hand the model one made-up flower and ask for its species. Feature order is: `[sepal length, sepal width, petal length, petal width]`.

In [ ]:
# A flower with small petals -> we expect 'setosa'
new_flower = np.array([[5.0, 3.4, 1.5, 0.2]])
new_flower_scaled = scaler.transform(new_flower)

pred_code = clf.predict(new_flower_scaled)[0]
print("Input measurements:", new_flower.ravel())
print("Predicted species: ", target_names[pred_code])

### Example B — multiple predictions at once

We can classify a whole batch in a single call. Each row is one flower.

In [ ]:
new_flowers = np.array([
    [5.1, 3.5, 1.4, 0.2],   # tiny petals   -> expect setosa
    [6.0, 2.7, 4.5, 1.5],   # medium petals -> expect versicolor
    [6.7, 3.0, 5.9, 2.1],   # large petals  -> expect virginica
])
new_flowers_scaled = scaler.transform(new_flowers)

batch_preds = clf.predict(new_flowers_scaled)
for measurements, code in zip(new_flowers, batch_preds):
    print(f"{measurements}  ->  {target_names[code]}")

### How to interpret the results

- The model outputs a **class code** (`0/1/2`); we map it back to a readable species name via `target_names`.
- Predictions match our expectations because petal size is the strongest signal for species — exactly what the distribution and scatter plots showed earlier.
- KNN gives a **hard label** (the majority vote). If you needed a **confidence score**, you could report the *fraction of the `k` neighbors* that voted for the winning class (see the exercises).

## 14. Common mistakes and troubleshooting

| Problem | Cause | Fix |
|--------|-------|-----|
| One feature dominates predictions | Features on different scales | **Standardize** with `StandardScaler` (we did this) |
| Suspiciously perfect accuracy | Test data leaked into training / scaler fit on all data | Fit the scaler on **train only**; keep the test set untouched |
| Ties in the vote | Even `k` with two classes | Prefer an **odd `k`**, or break ties by nearest distance |
| Very slow on big datasets | KNN compares to every training point | Use KD-/Ball-trees (scikit-learn) or reduce data size |
| Accuracy swings a lot between runs | Small test set + a single random split | Use **cross-validation** for a stable estimate |
| Poor results in high dimensions | *Curse of dimensionality* — distances lose meaning | Reduce features (e.g. **PCA**) or select the informative ones |
| Animation is slow or doesn't show | Heavy grid / missing display | Use a coarser grid, fewer frames, and `HTML(anim.to_jshtml())` |

## 15. Key takeaways

- **KNN is instance-based and lazy**: it memorizes the training set and does all the work at prediction time.
- A prediction is just **"find the `k` closest points and take a majority vote."**
- **Scaling matters** because KNN relies on distances.
- **`k` trades off** overfitting (small `k`) against underfitting (large `k`); tune it with a validation set or cross-validation.
- Evaluate with more than accuracy: **precision, recall, F1, and the confusion matrix** reveal *how* the model succeeds or fails.
- Despite its simplicity, KNN can be very effective on well-separated data like Iris.

## 16. Exercises

Try these to deepen your understanding (all doable with the code above):

1. **Prediction confidence.** Modify `_predict` to also return the fraction of the `k` neighbors that voted for the winning class, and print it in Example A.
2. **Distance metric.** Add a **Manhattan** distance option ($\sum |x_1 - x_2|$) and compare accuracy to Euclidean.
3. **No scaling.** Re-run training on the *unscaled* `X_train` and compare accuracy — does it change much for Iris? Why?
4. **Cross-validation.** Replace the single split with 5-fold cross-validation and report the mean ± std accuracy for a few `k` values.
5. **Two features only.** Retrain on just the two sepal features and inspect how much accuracy drops. Which features carry the signal?
6. **Weighted voting.** Give closer neighbors more influence (weight $\propto 1/\text{distance}$) and see whether it helps.
7. **Animate your query.** Change the `query` point in the Section 12.2 animation and watch how the vote — and the final class — changes as it moves across the border.

## 17. Next steps and improvements

- **Compare against `sklearn.neighbors.KNeighborsClassifier`** to validate your implementation and see the speed difference.
- **Vectorize** the distance computation with broadcasting to make prediction much faster.
- Try KNN on a **harder dataset** (e.g. digits, wine) to feel the *curse of dimensionality*.
- Explore other simple classifiers in this folder — logistic regression, decision trees, naive Bayes — and compare their decision boundaries on Iris.

### Saving an animation as a GIF (optional)

The inline `to_jshtml()` player is the easiest option. If you'd like a shareable **GIF**, save any of the animations above with the Pillow writer (no `ffmpeg` needed):

```python
# anim is the FuncAnimation object from Section 12.1
anim.save(\"knn_boundary.gif\", writer=\"pillow\", fps=1)
```

For MP4 you'd install `ffmpeg` and use `writer=\"ffmpeg\"`. For polished, presentation-grade math animations you could also explore **Manim**, though it needs a heavier setup (Cairo/LaTeX/ffmpeg) and renders to video files rather than inline notebook players.

---
*Built from the `knn.py` \"ML from scratch\" implementation. The KNN algorithm is preserved as in the original; the surrounding tutorial — explanations, visuals, animations, scaling, evaluation, and exercises — was added for self-study.*